In [1]:
import sys, os, shutil, tempfile, warnings
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

NOTEBOOK_DIR = Path().resolve()                    # tests/
SRC_DIR      = NOTEBOOK_DIR.parent / "src"         # src/

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# ── Imports ───────────────────────────────────────────────────────────────────
from dataloaders._ts_dataloader      import DataLoaderFactory
from dataloaders.ts_sharding        import (
    write_sharded_dataset,
    ShardedTrainDataset,
    ShardedValDataset,
    ShardedTestDataset,
)
from common.train import train
from models.momentmica import MOMENT
from losses.torch_losses import get_loss

warnings.filterwarnings("ignore")
print("imports OK")

/home/wpotosna/miniconda3/envs/neuralforecast/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


imports OK


In [3]:
# ── Synthetic dataframe factory ───────────────────────────────────────────────

def make_df(
    n_series: int = 3,
    n_steps:  int = 500,
    n_hist:   int = 1,       # number of hist exog cols
    seed:     int = 42,
) -> pd.DataFrame:
    """
    Returns a long-format DataFrame with columns:
        unique_id, ds, y, available_mask, hist_0, hist_1, ...
    All series have equal length (n_steps timesteps).
    available_mask is 1 everywhere (no gaps).
    """
    rng   = np.random.default_rng(seed)
    times = pd.date_range("2020-01-01", periods=n_steps, freq="5min")
    rows  = []
    for s in range(n_series):
        y    = rng.standard_normal(n_steps).astype(np.float32)
        hist = {f"hist_{i}": rng.standard_normal(n_steps).astype(np.float32)
                for i in range(n_hist)}
        df_s = pd.DataFrame({"unique_id": f"series_{s}", "ds": times, "y": y,
                              "available_mask": np.ones(n_steps, dtype=np.float32),
                              **hist})
        rows.append(df_s)
    return pd.concat(rows, ignore_index=True)


# ── Config factories ──────────────────────────────────────────────────────────

def make_mcfg(
    context_length:    int  = 64,
    fcd_samples:       int  = 4,
    batch_size:        int  = 2,
    max_steps:         int  = 20,
    val_check_interval:int  = 10,
    mixing_strategy:   str  = "concat",
    normalize:         bool = False,
    checkpoint_dir:    str  = "/tmp/ckpts",
    num_workers:       int  = 0,
    transformer_backbone: str = "google/t5-efficient-tiny",
):
    return SimpleNamespace(
        context_length         = context_length,
        patch_len              = 8,
        stride                 = 1,
        h                      = 6,
        pe_type                = 'sincos',
        learn_pe               = False,
        dropout                = 0.0,
        transformer_backbone   = transformer_backbone,
        infini_mixer_type      = "none",
        infini_channel_exclusion = False,
        layerwise_beta         = False,
        channelwise_beta       = False,
        mlpmixer_hidden_size   = 128,
        mlpmixer_n_layers      = 3,
        mlpmixer_dropout       = 0.0,
        multivariate_head      = False,
        head_dropout           = 0.0,
        hidden_size            = 256,
        linear_hidden_size     = 1024,
        n_heads                = 4,
        n_layers               = 4,
        learning_rate          = 1e-3,
        fcd_samples            = fcd_samples,
        batch_size             = batch_size,
        valid_batch_size       = batch_size,
        max_steps              = max_steps,
        val_check_interval     = val_check_interval,
        gradient_clip_val      = 1.0,
        early_stopping_patience= 9999,  # disable for tests
        monitor_metric         = "loss",
        monitor_mode           = "min",
        mixing_strategy        = mixing_strategy,
        drop_last              = False,
        batch_mode             = "full_series",
        checkpoint_dir         = checkpoint_dir,
        checkpoint_step        = 99999,  # suppress mid-run saves
        num_workers            = num_workers,
        revin                  = True,
        revin_affine           = False,
        revin_subtract_last    = False,
        loss                   = "mae",
    )


def make_entry(
    path:            str,
    name:            str  = "ds",
    horizon:         int  = 6,
    val_size:        int  = 50,
    test_size:       int  = 50,
    weight:          float= 1.0,
    hist_exog_cols:  list = None,
    futr_exog_cols:  list = None,
    stat_exog_cols:  list = None,
    per_series_split:bool = False,
    use_context_head:bool = True,
    sharded_dir:     str  = None,
):
    return SimpleNamespace(
        path             = path,
        name             = name,
        horizon          = horizon,
        val_size         = val_size,
        test_size        = test_size,
        weight           = weight,
        hist_exog_cols   = hist_exog_cols  or [],
        futr_exog_cols   = futr_exog_cols  or [],
        stat_exog_cols   = stat_exog_cols  or [],
        per_series_split = per_series_split,
        use_context_head = use_context_head,
        sharded_dir      = sharded_dir,
    )


def make_dcfg(train_entries, val_entries=None, test_entries=None):
    return SimpleNamespace(
        train      = train_entries,
        validation = val_entries  or train_entries,
        test       = test_entries or train_entries,
    )

print("helpers OK")

helpers OK


## MICA - T5 Encoder 

In [2]:
print("=" * 60)
print("TEST — resume training from checkpoint")
print("=" * 60)

with tempfile.TemporaryDirectory() as tmpdir:
    csv_path = f"{tmpdir}/data.csv"
    make_df(n_series=3, n_steps=300).to_csv(csv_path, index=False)

    mcfg = make_mcfg(
        context_length=32, fcd_samples=4, batch_size=2,
        max_steps=10, checkpoint_dir=tmpdir,
    )
    entry = make_entry(csv_path, name="resume_ds", horizon=6, val_size=50, test_size=50)
    dcfg  = make_dcfg([entry])

    # First run — saves final.pt
    factory      = DataLoaderFactory(mcfg, dcfg)
    train_loader = factory.train_dataloader()
    val_loaders  = factory.val_dataloaders()

    mcfg.loss = get_loss(mcfg.loss)
    loss_fn = mcfg.loss
    model = MOMENT(mcfg)
    train(model, mcfg, train_loader, val_loaders, device=torch.device("cpu"), loss_fn=loss_fn)
    step_after_first = model.global_step
    print(f"after first run: global_step={step_after_first}")

    # Second run — resume from final.pt, train 10 more steps
    mcfg2 = make_mcfg(
        context_length=32, fcd_samples=4, batch_size=2,
        max_steps=step_after_first + 10, checkpoint_dir=tmpdir,
    )
    factory2     = DataLoaderFactory(mcfg2, dcfg)
    model2       = MOMENT(mcfg)
    train(
        model2, mcfg2,
        factory2.train_dataloader(), factory2.val_dataloaders(),
        device  = torch.device("cpu"),
        resume  = f"{tmpdir}/final.pt",
        loss_fn=loss_fn
    )
    print(f"after resume:    global_step={model2.global_step}")
    assert model2.global_step == step_after_first + 10, \
        f"expected {step_after_first + 10}, got {model2.global_step}"

print("\n✓ TEST PASSED")

TEST — resume training from checkpoint


NameError: name 'make_df' is not defined

## MICA - TST Encoder

In [ ]:
print("=" * 60)
print("TEST — resume training from checkpoint")
print("=" * 60)

with tempfile.TemporaryDirectory() as tmpdir:
    csv_path = f"{tmpdir}/data.csv"
    make_df(n_series=3, n_steps=300).to_csv(csv_path, index=False)

    mcfg = make_mcfg(
        context_length=32, fcd_samples=4, batch_size=2,
        max_steps=10, transformer_backbone='patchtst',
        checkpoint_dir=tmpdir,
    )
    mcfg.res_attention = True
    mcfg.d_k = 32
    mcfg.d_v = 32
    mcfg.qkv_bias = False
    mcfg.attn_dropout = False
    mcfg.proj_dropout = 0.0
    mcfg.norm = "BatchNorm"
    mcfg.activation = 'gelu'
    mcfg.pre_norm = False
    mcfg.store_attn = False

    entry = make_entry(csv_path, name="resume_ds", horizon=6, val_size=50, test_size=50)
    dcfg  = make_dcfg([entry])

    # First run — saves final.pt
    factory      = DataLoaderFactory(mcfg, dcfg)
    train_loader = factory.train_dataloader()
    val_loaders  = factory.val_dataloaders()

    mcfg.loss = get_loss(mcfg.loss)
    loss_fn = mcfg.loss
    model = MOMENT(mcfg)
    train(model, mcfg, train_loader, val_loaders, device=torch.device("cpu"), loss_fn=loss_fn)
    step_after_first = model.global_step
    print(f"after first run: global_step={step_after_first}")

    # Second run — resume from final.pt, train 10 more steps
    mcfg2 = make_mcfg(
        context_length=32, fcd_samples=4, batch_size=2,
        max_steps=step_after_first + 10, checkpoint_dir=tmpdir,
    )
    factory2     = DataLoaderFactory(mcfg2, dcfg)
    model2       = MOMENT(mcfg)
    train(
        model2, mcfg2,
        factory2.train_dataloader(), factory2.val_dataloaders(),
        device  = torch.device("cpu"),
        resume  = f"{tmpdir}/final.pt",
        loss_fn=loss_fn
    )
    print(f"after resume:    global_step={model2.global_step}")
    assert model2.global_step == step_after_first + 10, \
        f"expected {step_after_first + 10}, got {model2.global_step}"

print("\n✓ TEST PASSED")

07:50:20  INFO      common._base_model  Training — max_steps=10  val_every=10  patience=9999  device=cpu


TEST 6 — resume training from checkpoint


Training:  40%|████      | 4/10 [00:00<00:00, 11.12it/s, train=1.9675]

enc_out.shape=torch.Size([3, 28, 256])
enc_out_windows.shape=torch.Size([1, 3, 4, 25, 256])
forecast.shape=torch.Size([1, 3, 4, 6])
enc_out.shape=torch.Size([3, 28, 256])
enc_out_windows.shape=torch.Size([1, 3, 4, 25, 256])
forecast.shape=torch.Size([1, 3, 4, 6])
enc_out.shape=torch.Size([3, 28, 256])
enc_out_windows.shape=torch.Size([1, 3, 4, 25, 256])
forecast.shape=torch.Size([1, 3, 4, 6])
enc_out.shape=torch.Size([3, 28, 256])
enc_out_windows.shape=torch.Size([1, 3, 4, 25, 256])
forecast.shape=torch.Size([1, 3, 4, 6])
enc_out.shape=torch.Size([3, 28, 256])
enc_out_windows.shape=torch.Size([1, 3, 4, 25, 256])
forecast.shape=torch.Size([1, 3, 4, 6])
enc_out.shape=torch.Size([3, 28, 256])
enc_out_windows.shape=torch.Size([1, 3, 4, 25, 256])
forecast.shape=torch.Size([1, 3, 4, 6])


Training: 100%|██████████| 10/10 [00:00<00:00, 15.16it/s, train=1.2967, val=1.7134]
07:50:21  INFO      common.train  Training complete.


...

enc_out.shape=torch.Size([3, 28, 256])
enc_out_windows.shape=torch.Size([1, 3, 4, 25, 256])
forecast.shape=torch.Size([1, 3, 4, 6])
enc_out.shape=torch.Size([3, 28, 256])
enc_out_windows.shape=torch.Size([1, 3, 4, 25, 256])
forecast.shape=torch.Size([1, 3, 4, 6])
enc_out.shape=torch.Size([3, 28, 256])
enc_out_windows.shape=torch.Size([1, 3, 4, 25, 256])
forecast.shape=torch.Size([1, 3, 4, 6])


Training: 100%|██████████| 20/20 [00:00<00:00, 26.60it/s, train=1.0643, val=1.6250]
07:50:22  INFO      common.train  Training complete.


enc_out.shape=torch.Size([3, 28, 256])
enc_out_windows.shape=torch.Size([1, 3, 4, 25, 256])
forecast.shape=torch.Size([1, 3, 4, 6])
enc_out.shape=torch.Size([3, 69, 256])
enc_out_windows.shape=torch.Size([1, 3, 45, 25, 256])
forecast.shape=torch.Size([1, 3, 45, 6])


07:50:22  INFO      common._base_model  Trainer state saved → /tmp/tmp2tr13yn5/final.pt
07:50:22  INFO      common.train  Saved → /tmp/tmp2tr13yn5/final.pt


after resume:    global_step=20

✓ TEST PASSED
